In [ ]:
import librosa
import pandas as pd
import os
import json

In [132]:
TARGET_DURATION = 4.0
SR = 16000
target_length = int(TARGET_DURATION * SR)
base_dataset_path = "../DS_10283_3336/LA/"

In [133]:
def get_files_relationship(path):
    train_prot = pd.read_csv(path, sep=" ", header=None)
    data = {}

    for i in train_prot.iloc:
        data[i[1]] = i[4] == "bonafide"

    return data

In [134]:
def serialize_array(array):
    return json.dumps(array.tolist())

In [135]:
def normalize_audio(path):
    y, sr = librosa.load(path, sr=SR, duration=TARGET_DURATION)

    if len(y) < target_length:
        y = librosa.util.pad_center(y, size=target_length)
    
    return y, sr

In [136]:
def extract_audio_features(path, relationship):
    basename = os.path.basename(path).split(".")[0]
    y, sr = normalize_audio(path)
    mcff = librosa.feature.mfcc(y=y, sr=sr)
    sc = librosa.feature.spectral_centroid(y=y, sr=sr)
    sro = librosa.feature.spectral_rolloff(y=y, sr=sr)
    cstft = librosa.feature.chroma_stft(y=y, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y=y)
    rms = librosa.feature.rms(y=y)
    is_bonafide = relationship.get(basename)
    
    if is_bonafide is None:
        return None

    return {
        basename: {
            "mcff": mcff,
            "sc": sc,
            "sro": sro,
            "cstft": cstft,
            "zcr": zcr,
            "rms": rms,
            "is_bonafide": is_bonafide
        }
    }

In [137]:
def convert_dict_to_dataframe(data):
    rows = []

    for filename, value in data.items():
        rows.append({
            "filename": filename,
            "mcff": serialize_array(value["mcff"]),
            "sc": serialize_array(value["sc"]),
            "sro": serialize_array(value["sro"]),
            "cstft": serialize_array(value["cstft"]),
            "zcr": serialize_array(value["zcr"]),
            "rms": serialize_array(value["rms"]),
            "is_bonafide": value["is_bonafide"]
        })

    return pd.DataFrame(rows)

In [138]:
# TRAIN
full_path = f"{base_dataset_path}ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
train_relationship = get_files_relationship(full_path)

train_data = {}

for filename in os.listdir(f"{base_dataset_path}ASVspoof2019_LA_train/flac"):
    file_data = extract_audio_features(f"{base_dataset_path}ASVspoof2019_LA_train/flac/{filename}", train_relationship)
    if file_data is not None:
        train_data.update(file_data)

train_df = convert_dict_to_dataframe(train_data)
train_df.to_csv("../Data/train_data_features.csv", index=False)

In [139]:
# DEV
full_path = f"{base_dataset_path}ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
dev_relationship = get_files_relationship(full_path)

dev_data = {}

for filename in os.listdir(f"{base_dataset_path}ASVspoof2019_LA_dev/flac"):
    file_data = extract_audio_features(f"{base_dataset_path}ASVspoof2019_LA_dev/flac/{filename}", dev_relationship)
    if file_data is not None:
        dev_data.update(file_data)

dev_df = convert_dict_to_dataframe(dev_data)
dev_df.to_csv("../Data/dev_data_features.csv", index=False)

In [140]:
# EVAL
full_path = f"{base_dataset_path}ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"
eval_relationship = get_files_relationship(full_path)

eval_data = {}

for filename in os.listdir(f"{base_dataset_path}ASVspoof2019_LA_eval/flac"):
    file_data = extract_audio_features(f"{base_dataset_path}ASVspoof2019_LA_eval/flac/{filename}", eval_relationship)
    if file_data is not None:
        eval_data.update(file_data)

eval_df = convert_dict_to_dataframe(eval_data)
eval_df.to_csv("../Data/eval_data_features.csv", index=False)